# B2-019 — Session 5: Transformer Blocks and Architecture

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Position-wise feed-forward sublayer

The same two-layer MLP is applied independently to every token: $\operatorname{FFN}(x)=W_2\phi(W_1x+b_1)+b_2$. It changes features, not sequence positions.

**Checkpoint 1A.** Which dimensions are shared across tokens?

**Checkpoint 1B.** Why is this called position-wise?

In [ ]:
import torch
from torch import nn
SEED = 20260808
torch.manual_seed(SEED)
x = torch.arange(24, dtype=torch.float32).reshape(2, 3, 4)
ffn = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4))
assert ffn(x).shape == x.shape

## 2. Pre-norm residual ordering

A pre-norm block computes $y=x+\operatorname{MHA}(\operatorname{LN}(x))$, then $z=y+\operatorname{FFN}(\operatorname{LN}(y))$. Each residual adds tensors of identical shape.

**Worked example 1.** If a sublayer returns zero, its residual path returns the input exactly.

**Checkpoint 2A.** Which tensor is normalized before the second sublayer?

**Checkpoint 2B.** Where is the mask passed?

In [ ]:
def prenorm_trace(x, attention, ffn, norm1, norm2, mask):
    y = x + attention(norm1(x), mask=mask)
    z = y + ffn(norm2(y))
    return z

## 3. Encoder, decoder, and cross-attention roles

Encoder self-attention uses encoder states for Q, K, and V and usually a padding mask. Decoder causal self-attention uses decoder states for all three. Cross-attention uses decoder states for Q and encoder outputs for K and V.

**Worked example 2.** If decoder length is 4 and encoder length is 7, cross-attention scores have shape `(B,h,4,7)`.

**Checkpoint 3A.** Which length controls score rows?

**Checkpoint 3B.** Which length controls score columns?

## 4. Architecture trace

An encoder stack transforms a source sequence into contextual states. A decoder stack alternates causal self-attention, cross-attention, and feed-forward processing before a vocabulary projection. This unit studies mechanics; larger language-model objectives begin in B2-020.

**Checkpoint 4A.** Does an encoder-only block require a causal mask by definition?

**Checkpoint 4B.** Which sublayer changes neither batch nor sequence length?

## 5. Common pitfalls and completion

**Common pitfalls.** Broken: normalize the residual branch after overwriting its source. Fix: name intermediate `y` and follow equations. Broken: feed encoder outputs into decoder queries. Fix: Q names the destinations being updated.

**Exam connections.** Architecture questions reward explicit Q/K/V sources and a correct norm-residual trace, not a memorized diagram.

**Going deeper.** `B2-020-language-transformers` applies these blocks to language objectives after this unit is complete.

Checkpoint answers: 1A weights and biases; 1B no token mixing; 2A y; 2B attention sublayer; 3A query length; 3B key length; 4A no; 4B position-wise FFN.